In [7]:
'''Download a public dataset of online product reviews (for example, from Kaggle or UCI), split it into features and target, and use GridSearchCV to tune the 'max_depth' and 'n_estimators' hyperparameters of a RandomForestClassifier to predict if a review is positive or negative.'''

import pandas as pd

# Load Amazon product reviews
df = pd.read_csv(
    "amazon_cells_labelled.txt",
    sep="\t",
    header=None,
    names=["review", "sentiment"]
)

print(df.head())
print("Shape:", df.shape)

                                              review  sentiment
0  So there is no way for me to plug it in here i...          0
1                        Good case, Excellent value.          1
2                             Great for the jawbone.          1
3  Tied to charger for conversations lasting more...          0
4                                  The mic is great.          1
Shape: (1000, 2)


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
X = df["review"]
y = df["sentiment"]
vectorizer = TfidfVectorizer()

X_tfidf = vectorizer.fit_transform(X)

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (1000, 1847)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

model = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20]
}

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

Best Parameters: {'max_depth': None, 'n_estimators': 200}
Best CV Accuracy: 0.7924999999999999


In [12]:
'''Use RandomizedSearchCV to tune the 'C' and 'gamma' parameters of an SVM classifier on a dataset of your choice (such as predicting if an Instagram post will get high engagement based on features like caption length, time of posting, and hashtag count).<br><br><em><strong>Hint:</strong> Limit the number of iterations in RandomizedSearchCV to 10 for faster results.</em>'''

import pandas as pd

# Create a small Instagram engagement dataset
data = {
    "caption_length": [20, 150, 30, 200, 50, 180, 25, 220, 70, 160,
                       40, 190, 35, 210, 60, 170, 45, 230, 80, 140],
    
    "time_of_posting": [9, 20, 10, 21, 8, 19, 11, 20, 9, 18,
                        10, 21, 8, 20, 11, 19, 9, 21, 10, 18],
    
    "hashtag_count": [2, 15, 3, 18, 5, 14, 2, 20, 6, 13,
                      4, 17, 3, 19, 5, 15, 2, 21, 7, 12],
    
    # 1 = High engagement, 0 = Low engagement
    "high_engagement": [0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
                        0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)

print(df.head())

X = df[["caption_length", "time_of_posting", "hashtag_count"]]
y = df["high_engagement"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV

model = SVC()

param_distributions = {
    "C": [0.01, 0.1, 1, 10, 100],
    "gamma": [0.001, 0.01, 0.1, 1, 10]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_scaled, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV Accuracy:")
print(random_search.best_score_)

from sklearn.metrics import accuracy_score, classification_report

best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test_scaled)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

   caption_length  time_of_posting  hashtag_count  high_engagement
0              20                9              2                0
1             150               20             15                1
2              30               10              3                0
3             200               21             18                1
4              50                8              5                0
Best Parameters:
{'gamma': 0.01, 'C': 10}

Best CV Accuracy:
1.0
Test Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



In [15]:
'''Pick a dataset with a clear class imbalance (for example, fraud detection or rare event prediction), and apply SMOTE to oversample the minority class before running hyperparameter tuning with GridSearchCV on a RandomForestClassifier.<br><br><em><strong>Hint:</strong> Use the imblearn library’s SMOTE class for oversampling.</em>'''

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from collections import Counter

data = load_breast_cancer()

X = data.data
y = data.target

print("Original class distribution:", Counter(y))

# Create an imbalanced dataset
# Keep all class 0 samples
# Keep only 50 samples from class 1
class_0 = y == 0
class_1 = y == 1

X_0 = X[class_0]
y_0 = y[class_0]

X_1 = X[class_1][:50]
y_1 = y[class_1][:50]

import numpy as np

X_imbalanced = np.vstack((X_0, X_1))
y_imbalanced = np.concatenate((y_0, y_1))

print("Imbalanced distribution:", Counter(y_imbalanced))

X_train, X_test, y_train, y_test = train_test_split(
    X_imbalanced,
    y_imbalanced,
    test_size=0.2,
    random_state=42,
    stratify=y_imbalanced
)

print("Training distribution:", Counter(y_train))
print("Testing distribution:", Counter(y_test))

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Before SMOTE:", Counter(y_train))
print("After SMOTE:", Counter(y_train_smote))

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

model = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_smote, y_train_smote)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV Accuracy:")
print(grid_search.best_score_)

from sklearn.metrics import accuracy_score, classification_report

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Original class distribution: Counter({np.int64(1): 357, np.int64(0): 212})
Imbalanced distribution: Counter({np.int64(0): 212, np.int64(1): 50})
Training distribution: Counter({np.int64(0): 169, np.int64(1): 40})
Testing distribution: Counter({np.int64(0): 43, np.int64(1): 10})
Before SMOTE: Counter({np.int64(0): 169, np.int64(1): 40})
After SMOTE: Counter({np.int64(0): 169, np.int64(1): 169})
Best Parameters:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}

Best CV Accuracy:
0.9822212467076383
Test Accuracy: 0.9811320754716981

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        43
           1       0.91      1.00      0.95        10

    accuracy                           0.98        53
   macro avg       0.95      0.99      0.97        53
weighted avg       0.98      0.98      0.98        53

